# 02 — Audio Transcription & ASR Evaluation
## Maharah GDP | Call Center Agent Performance Analysis

**Prerequisite:** Stage 01 metadata/EDA completed successfully. Its verified results: **873 calls, 14 accents, 16 domains, 94,679 reference segments**, with 12 deterministically selected calls for our experiment.

**Goal of this notebook:**
1. Recreate or reuse the exact 12-call manifest from Stage 01 (seed 42, pinned AppTek dataset revision).
2. Run a **real ASR model on the original merged audio**, without passing reference text to the model.
3. Save time-stamped model-generated transcripts and word-level timestamps.
4. Compare the ASR output with independent reference transcripts using WER/CER and show qualitative errors.
5. Produce clean, resumable results for speaker diarization, summary and sentiment notebooks.

> **Methodology boundary:** AppTek `segments[].text`, `segments[].role`, `speaker_id` and timing are **dataset gold references**. We use the reference **text ONLY AFTER transcription to evaluate WER**. The output of this notebook does **not** include independently inferred customer/agent identities. We will run/evaluate actual diarization in the next stage, rather than claim the reference labels were predicted by our model.

**Dataset:** [AppTek Call-Center Dialogues](https://huggingface.co/datasets/apptek-com/apptek_callcenter_dialogues), **`diarization`** config. Data are simulated role-play, not genuine private customer calls; respect the CC BY-SA 4.0 terms and evaluation-only intended use. Do not upload the original WAV files to GitHub.

**Model:** [SYSTRAN faster-whisper / Whisper small.en](https://huggingface.co/Systran/faster-whisper-small.en). We pin model and dataset commit SHAs for reproducibility. Model choice is a baseline to **measure**, not an assertion that this is the optimal ASR model.

**Colab CPU-only:** Runtime → Change runtime type → **CPU** (no GPU quota consumed). This notebook starts with **only 2 pilot calls**; move to all 12 only after inspecting the results. An internet connection is required in Colab.

**Why CPU / small.en?** Small English + int8 is a practical no-GPU baseline; it may be less accurate than medium.en, so WER/CER must determine whether an upgrade is necessary. The model still produces word timestamps and uses the same dataset / evaluation method.


### Expected artifacts

- `data/processed/selected_calls.csv`: manifest reproduced with the same Stage 01 logic (or reused if already present).
- `data/processed/asr/small_en_cpu/CALL_001.json` etc.: **predicted** transcript, timestamped model segments and words.
- `data/processed/asr/small_en_cpu/metrics.csv`: per-call WER, CER, processing speed, accent and domain.
- `data/processed/asr/small_en_cpu/run_config.json`: pinned model/data versions, settings, normalization policy.

**Note:** If you changed the sampling logic after Stage 01, upload your original `selected_calls.csv` into `data/processed/` rather than letting this notebook regenerate it.

In [11]:
%pip -q install "faster-whisper>=1.1,<2" "huggingface_hub>=0.26,<2" "pandas>=2,<3" "jiwer>=3,<5" "soundfile>=0.12" "tqdm>=4.60"

In [13]:
from pathlib import Path
from datetime import datetime, timezone
import json, posixpath, re, time, platform, unicodedata
import pandas as pd
import soundfile as sf
import jiwer
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from faster_whisper import WhisperModel
from tqdm.auto import tqdm
from IPython.display import display

# Reproduce exactly the same AppTek revision used in Stage 01.
REPO_ID = "apptek-com/apptek_callcenter_dialogues"
DATASET_REVISION = "b98967d9946f7f59f58d08624a2a00fe98fe0219"
CONFIG = "diarization"
MODEL_REPO = "Systran/faster-whisper-small.en"
MODEL_TAG = "small_en_cpu"

# IMPORTANT: run the first two calls before switching this flag to False.
PILOT_ONLY = False
PILOT_COUNT = 2
SEED = 42
ACCENTS_TARGET = ["en-US_General", "en-GB", "en-IN", "en-AU"]
CALLS_PER_ACCENT = 3

ROOT = Path("data")
META_DIR = ROOT / "metadata"
RAW_DIR = ROOT / "raw_samples"
OUT_DIR = ROOT / "processed"
ASR_DIR = OUT_DIR / "asr" / MODEL_TAG
for p in [META_DIR, RAW_DIR, OUT_DIR, ASR_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = "cpu"
COMPUTE_TYPE = "int8"
print("Device:", DEVICE, "| Compute type:", COMPUTE_TYPE)
print("Working directory:", Path.cwd().resolve())
print("Dataset revision pinned:", DATASET_REVISION)
print("Pilot mode:", PILOT_ONLY)

Device: cpu | Compute type: int8
Working directory: /content
Dataset revision pinned: b98967d9946f7f59f58d08624a2a00fe98fe0219
Pilot mode: False


## 1. Recover the same exact selected calls (no downloading all WAVs)

If the Stage 01 output `data/processed/selected_calls.csv` exists in this runtime, this notebook uses it. Otherwise it reproduces the original deterministic selection using the four accent metadata groups only. This makes it possible to run the notebook in a **new Colab runtime** without re-running the full dataset EDA, while preserving the sample-selection policy.

We still load each selected call's reference metadata only to **hold it for later independent evaluation**; it is never fed to the ASR model.

In [14]:
api = HfApi()
repo_files = api.list_repo_files(REPO_ID, repo_type="dataset", revision=DATASET_REVISION)
metadata_paths = sorted(p for p in repo_files if p.startswith(f"{CONFIG}/") and p.endswith("/metadata.jsonl"))
metadata_paths = [p for p in metadata_paths if p.split("/")[1] in ACCENTS_TARGET]
assert len(metadata_paths) == len(ACCENTS_TARGET), (metadata_paths, ACCENTS_TARGET)
print("Metadata groups used:", metadata_paths)

all_rows = []
for meta_path in tqdm(metadata_paths, desc="Download four small metadata files"):
    local_meta = hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset", filename=meta_path,
        revision=DATASET_REVISION, local_dir=str(META_DIR)
    )
    with open(local_meta, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                item["_metadata_path"] = meta_path
                all_rows.append(item)

# Lookup is used to evaluate predictions AFTER the model completes transcription.
lookup = {(r["_metadata_path"], r["file_name"]): r for r in all_rows}
manifest_path = OUT_DIR / "selected_calls.csv"
if manifest_path.exists():
    sample = pd.read_csv(manifest_path)
    print("Using an existing Stage 01 manifest:", manifest_path)
else:
    import random
    rng = random.Random(SEED)
    rows = []
    for item in all_rows:
        rows.append({
            "file_name": item["file_name"], "accent": item["accent"],
            "domain": item["domain"], "duration_sec": float(item["duration"]),
            "segments": len(item["segments"]),
            "customer_segments": sum(s["role"] == "customer" for s in item["segments"]),
            "agent_segments": sum(s["role"] == "agent" for s in item["segments"]),
            "metadata_path": item["_metadata_path"],
        })
    metadata_df = pd.DataFrame(rows)
    selected = []
    for accent in ACCENTS_TARGET:
        group = metadata_df[metadata_df.accent == accent]
        domains = sorted(group.domain.dropna().unique().tolist())
        assert len(domains) >= CALLS_PER_ACCENT
        rng.shuffle(domains)
        for domain in domains[:CALLS_PER_ACCENT]:
            candidates = group[group.domain == domain].sort_values("file_name")
            selected.append(candidates.iloc[rng.randrange(len(candidates))].to_dict())
    sample = pd.DataFrame(selected)
    sample.insert(0, "call_id", [f"CALL_{i:03d}" for i in range(1, len(sample) + 1)])
    sample.to_csv(manifest_path, index=False)
    print("Manifest recreated from pinned metadata + seed 42:", manifest_path)

required = {"call_id", "metadata_path", "file_name", "accent", "domain", "duration_sec"}
assert required.issubset(sample.columns), sorted(required - set(sample.columns))
assert len(sample) == 12 and sample.call_id.is_unique
assert all((r.metadata_path, r.file_name) in lookup for r in sample.itertuples()), (
    "Manifest did not match the pinned metadata; check your Stage 01 sampling/version."
)
print("Selected calls:", len(sample), "| Accents:", sample.accent.nunique(), "| Domains:", sample.domain.nunique())
display(sample[["call_id", "accent", "domain", "duration_sec"]].round(2))
# Stage 01's first item according to the report the user shared:
assert sample.iloc[0]["file_name"] == "audio/en_US_General_Food_1586724.wav", (
    "The first sampled file differs from Stage 01. Stop and inspect before continuing."
)

Metadata groups used: ['diarization/en-AU/metadata.jsonl', 'diarization/en-GB/metadata.jsonl', 'diarization/en-IN/metadata.jsonl', 'diarization/en-US_General/metadata.jsonl']


Download four small metadata files:   0%|          | 0/4 [00:00<?, ?it/s]

Using an existing Stage 01 manifest: data/processed/selected_calls.csv
Selected calls: 12 | Accents: 4 | Domains: 8


,call_id,accent,domain,duration_sec
0,CALL_001,en-US_General,food,386.64
1,CALL_002,en-US_General,hospitality,630.75
2,CALL_003,en-US_General,entertainment,701.91
3,CALL_004,en-GB,hospitality,421.52
4,CALL_005,en-GB,retail,611.11
5,CALL_006,en-GB,entertainment,603.48
6,CALL_007,en-IN,telecom,631.71
7,CALL_008,en-IN,insurance,625.66
8,CALL_009,en-IN,health,422.21
9,CALL_010,en-AU,banking,420.00


## 2. Download only the pilot WAVs and perform audio integrity checks

These are **merged, single-channel** waveforms containing both agent and customer. The reference annotations are kept separately for later scoring.

In [15]:
to_process = sample.head(PILOT_COUNT).copy() if PILOT_ONLY else sample.copy()
audio_paths = {}
for row in tqdm(to_process.itertuples(index=False), total=len(to_process), desc="Get selected WAVs"):
    repo_path = posixpath.normpath(posixpath.join(posixpath.dirname(row.metadata_path), row.file_name))
    assert repo_path in repo_files, f"Audio missing from pinned repository: {repo_path}"
    local_audio = hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset", filename=repo_path,
        revision=DATASET_REVISION, local_dir=str(RAW_DIR)
    )
    audio = sf.info(local_audio)
    assert audio.channels == 1, f"Expected mixed mono audio: {row.call_id}"
    assert audio.samplerate == 16000, f"Unexpected sampling rate {audio.samplerate}: {row.call_id}"
    assert abs(audio.duration - float(row.duration_sec)) < 2.0
    audio_paths[row.call_id] = str(local_audio)
    print(f"{row.call_id}: {audio.duration:.1f}s | {audio.samplerate} Hz | mono | {row.accent} | {row.domain}")

Get selected WAVs:   0%|          | 0/12 [00:00<?, ?it/s]

CALL_001: 386.6s | 16000 Hz | mono | en-US_General | food
CALL_002: 630.7s | 16000 Hz | mono | en-US_General | hospitality


diarization/en-US_General/audio/en_US_Ge(…): reconstructing file:   0%|          |  0.00B / 22.5MB            

diarization/en-US_General/audio/en_US_Ge(…): downloading bytes:           |  0.00B            

CALL_003: 701.9s | 16000 Hz | mono | en-US_General | entertainment


diarization/en-GB/audio/en_GB_Hospitalit(…): reconstructing file:   0%|          |  0.00B / 13.5MB            

... (22 outputs left out)

CALL_011: 803.9s | 16000 Hz | mono | en-AU | insurance


diarization/en-AU/audio/en_AU_Food_15934(…): reconstructing file:   0%|          |  0.00B / 27.9MB            

diarization/en-AU/audio/en_AU_Food_15934(…): downloading bytes:           |  0.00B            

CALL_012: 871.0s | 16000 Hz | mono | en-AU | food


## 3. Pin and initialize the ASR model

Use a **real pre-trained model**, running locally in your Colab CPU session. We record the exact Hugging Face model commit SHA instead of depending on a moving branch. We force English because **the benchmark is known to be in English**; that is a dataset-level fact rather than leakage of the words being transcribed.

In [16]:
model_revision = api.model_info(MODEL_REPO).sha
print("Model revision:", model_revision)
model_dir = snapshot_download(repo_id=MODEL_REPO, revision=model_revision)
model = WhisperModel(model_dir, device=DEVICE, compute_type=COMPUTE_TYPE)
ASR_SETTINGS = {
    "language": "en",
    "beam_size": 5,
    "vad_filter": True,
    "vad_parameters": {"min_silence_duration_ms": 2000},
    "word_timestamps": True,
    "condition_on_previous_text": False,
}
print("ASR model ready:", MODEL_REPO, "|", DEVICE, COMPUTE_TYPE)
print("Settings:", ASR_SETTINGS)

Model revision: d1d751a5f8271d482d14ca55d9e2deeebbae577f


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

ASR model ready: Systran/faster-whisper-small.en | cpu int8
Settings: {'language': 'en', 'beam_size': 5, 'vad_filter': True, 'vad_parameters': {'min_silence_duration_ms': 2000}, 'word_timestamps': True, 'condition_on_previous_text': False}


## 4. Transcribe the original WAV audio and save model predictions

**Strict no-leakage rule:** In this cell, only `audio_paths` and ASR configuration are used to generate text. Neither the reference `segments` nor `text` nor `role` is an input to `transcribe`.

`faster-whisper` returns a **generator**, so we explicitly materialize it. Each call saves its results immediately and is skipped on reruns if the saved output has the matching model/dataset revisions and settings. This makes the notebook recoverable after Colab interruptions.
**Compatibility fix (Colab, November 2026):** Some `faster-whisper` versions pass `metadata_errors` to PyAV 19, which removed that argument. WAV decoding is done with `soundfile` into a mono float32 NumPy array; Faster-Whisper accepts the array directly. This bypasses the incompatible PyAV audio-opening call while keeping the same model and CPU settings.


In [17]:
def serialize_segment(seg):
    """Convert model predictions (never dataset reference segments) to JSON-safe data."""
    words = []
    for w in (seg.words or []):
        words.append({
            "start": round(float(w.start), 3) if w.start is not None else None,
            "end": round(float(w.end), 3) if w.end is not None else None,
            "word": w.word,
            "probability": round(float(w.probability), 4) if w.probability is not None else None,
        })
    return {
        "start": round(float(seg.start), 3), "end": round(float(seg.end), 3),
        "text": seg.text.strip(), "words": words,
    }

for row in tqdm(to_process.itertuples(index=False), total=len(to_process), desc="ASR inference"):
    output_path = ASR_DIR / f"{row.call_id}.json"
    if output_path.exists():
        cached = json.loads(output_path.read_text(encoding="utf-8"))
        if (cached.get("model_revision") == model_revision and
            cached.get("dataset_revision") == DATASET_REVISION and
            cached.get("asr_settings") == ASR_SETTINGS):
            print(row.call_id, "already transcribed with same revisions/settings — skipping")
            continue
    tic = time.perf_counter()
    # Decode the 16-kHz mono WAV ourselves. Passing a NumPy array to Faster-Whisper
    # bypasses the PyAV 19 / faster-whisper metadata_errors incompatibility.
    waveform, sr = sf.read(audio_paths[row.call_id], dtype="float32")
    assert sr == 16000 and waveform.ndim == 1, (
        f"Expected 16-kHz mono audio for {row.call_id}; got sr={sr}, shape={waveform.shape}"
    )
    seg_iter, info = model.transcribe(waveform, **ASR_SETTINGS)
    predicted = [serialize_segment(s) for s in seg_iter]   # generator executed here
    elapsed = time.perf_counter() - tic
    transcription = " ".join(s["text"] for s in predicted if s["text"]).strip()
    result = {
        "call_id": row.call_id, "accent": row.accent, "domain": row.domain,
        "audio_duration_sec": round(float(row.duration_sec), 3),
        "source_dataset": REPO_ID, "dataset_revision": DATASET_REVISION,
        "model_repo": MODEL_REPO, "model_revision": model_revision,
        "asr_settings": ASR_SETTINGS, "run_utc": datetime.now(timezone.utc).isoformat(),
        "inference_seconds": round(elapsed, 2),
        "predicted_language": info.language,
        "predicted_transcript": transcription,
        "predicted_segments": predicted,
        "speaker_label_method": "Not inferred yet: model predictions contain no speaker IDs",
    }
    output_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"{row.call_id}: {len(predicted)} ASR segments | {len(transcription.split())} words | {elapsed:.1f}s | saved")

print("Completed ASR for:", [r.call_id for r in to_process.itertuples(index=False)])

ASR inference:   0%|          | 0/12 [00:00<?, ?it/s]

CALL_001 already transcribed with same revisions/settings — skipping
CALL_002 already transcribed with same revisions/settings — skipping
CALL_003: 95 ASR segments | 1087 words | 158.6s | saved
CALL_004: 75 ASR segments | 1077 words | 127.3s | saved
CALL_005: 132 ASR segments | 1603 words | 187.9s | saved
CALL_006: 82 ASR segments | 1130 words | 143.4s | saved
CALL_007: 86 ASR segments | 1100 words | 160.0s | saved
CALL_008: 91 ASR segments | 1238 words | 173.5s | saved
CALL_009: 53 ASR segments | 749 words | 110.5s | saved
CALL_010: 68 ASR segments | 872 words | 116.4s | saved
CALL_011: 118 ASR segments | 1810 words | 238.0s | saved
CALL_012: 190 ASR segments | 2282 words | 262.7s | saved
Completed ASR for: ['CALL_001', 'CALL_002', 'CALL_003', 'CALL_004', 'CALL_005', 'CALL_006', 'CALL_007', 'CALL_008', 'CALL_009', 'CALL_010', 'CALL_011', 'CALL_012']


## 5. Evaluate transcription accuracy against the gold transcripts

**WER** = (substitutions + deletions + insertions) / reference words. **CER** = character error rate. Lower is better; WER can exceed 100% on very poor predictions. The reference transcript is assembled in time order from dataset annotations **after** model generation.

**Important limitations:** our *lightweight* normalization lowercases and strips punctuation but preserves conversational filler words; it is **not** AppTek's full official scoring script. Therefore **do not compare these WERs numerically to published leaderboard scores**. Evaluate all our models under this same normalization policy. Mixed speech overlap can increase errors.

In [18]:
def normalize_for_metrics(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = text.replace("’", "'").replace("‘", "'")
    text = re.sub(r"[^a-z0-9'\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

metrics = []
normalized_refs, normalized_hyps = [], []
for row in to_process.itertuples(index=False):
    prediction = json.loads((ASR_DIR / f"{row.call_id}.json").read_text(encoding="utf-8"))
    reference_row = lookup[(row.metadata_path, row.file_name)]
    ref_segments = sorted(reference_row["segments"], key=lambda x: (float(x["start"]), float(x["end"])))
    reference = " ".join(str(s.get("text", "")) for s in ref_segments)
    ref_norm = normalize_for_metrics(reference)
    hyp_norm = normalize_for_metrics(prediction["predicted_transcript"])
    if not ref_norm:
        raise ValueError(f"Empty reference transcript for {row.call_id}")
    word_output = jiwer.process_words(ref_norm, hyp_norm)
    normalized_refs.append(ref_norm)
    normalized_hyps.append(hyp_norm)
    metrics.append({
        "call_id": row.call_id, "accent": row.accent, "domain": row.domain,
        "duration_min": round(float(row.duration_sec)/60, 2),
        "reference_words": len(ref_norm.split()),
        "predicted_words": len(hyp_norm.split()),
        "wer": round(float(word_output.wer), 4),
        "cer": round(float(jiwer.cer(ref_norm, hyp_norm)), 4),
        "substitutions": word_output.substitutions,
        "deletions": word_output.deletions,
        "insertions": word_output.insertions,
        "inference_sec": prediction["inference_seconds"],
        "real_time_factor": round(prediction["inference_seconds"] / float(row.duration_sec), 3),
    })

metrics_df = pd.DataFrame(metrics)
metrics_path = ASR_DIR / "metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Per-call ASR evaluation:")
display(metrics_df.round(3))
print(f"\nCORPUS (micro) WER: {jiwer.wer(normalized_refs, normalized_hyps):.3f}")
print(f"CORPUS (micro) CER: {jiwer.cer(normalized_refs, normalized_hyps):.3f}")
print(f"Mean per-call WER: {metrics_df.wer.mean():.3f} (different from corpus WER)")
print("Saved:", metrics_path)

Per-call ASR evaluation:


,call_id,accent,domain,duration_min,reference_words,predicted_words,wer,cer,substitutions,deletions,insertions,inference_sec,real_time_factor
0,CALL_001,en-US_General,food,6.44,789,753,0.085,0.044,23,40,4,97.07,0.251
1,CALL_002,en-US_General,hospitality,10.51,1616,1419,0.166,0.113,44,211,14,177.33,0.281
2,CALL_003,en-US_General,entertainment,11.70,1452,1143,0.257,0.224,52,315,6,158.57,0.226
3,CALL_004,en-GB,hospitality,7.03,1142,1091,0.153,0.107,68,79,28,127.27,0.302
4,CALL_005,en-GB,retail,10.19,1818,1609,0.309,0.219,231,270,61,187.90,0.307
5,CALL_006,en-GB,entertainment,10.06,1510,1135,0.362,0.278,126,398,23,143.41,0.238
6,CALL_007,en-IN,telecom,10.53,1261,1104,0.280,0.208,146,182,25,159.99,0.253
7,CALL_008,en-IN,insurance,10.43,1412,1248,0.239,0.177,123,189,25,173.51,0.277
8,CALL_009,en-IN,health,7.04,878,763,0.255,0.186,85,127,12,110.51,0.262
9,CALL_010,en-AU,banking,7.00,965,872,0.154,0.109,48,97,4,116.39,0.277



CORPUS (micro) WER: 0.207
CORPUS (micro) CER: 0.149
Mean per-call WER: 0.208 (different from corpus WER)
Saved: data/processed/asr/small_en_cpu/metrics.csv


## 6. Inspect a real reference-vs-prediction excerpt (qualitative validation)

Review **at least the first two cases manually**. Transcription may miss fillers, names, numbers, or fast/overlapping speech. WER is not a speaker diarization metric.

In [19]:
for row in to_process.head(2).itertuples(index=False):
    reference_row = lookup[(row.metadata_path, row.file_name)]
    reference = " ".join(s["text"] for s in sorted(reference_row["segments"], key=lambda x: x["start"]))
    prediction = json.loads((ASR_DIR / f"{row.call_id}.json").read_text(encoding="utf-8"))
    print("=" * 80)
    print(row.call_id, "|", row.accent, "|", row.domain)
    print("REFERENCE excerpt:", reference[:500])
    print("ASR PREDICTION excerpt:", prediction["predicted_transcript"][:500])
    print("NOTE: Reference content displayed for comparison ONLY; not used for ASR decoding.")

first_pred = json.loads((ASR_DIR / f"{to_process.iloc[0]['call_id']}.json").read_text(encoding="utf-8"))
word_examples = [w for seg in first_pred["predicted_segments"] for w in seg.get("words", [])]
display(pd.DataFrame(word_examples[:12]))

CALL_001 | en-US_General | food
REFERENCE excerpt: Ring ring, hi there, this is Leo from Feast and Co. your personal lunch superhero and I speaking with Ashley? Hi Leo, yeah this is Ashley I'm l~ looking into corporate catering for a team lunch. Ohh I love a good team lunch, nothing brings coworkers together like gourmet bites and good vibes. So tell me what kind of flavor adventure are we talking about here? Wow you sound very enthusiastic about this, I love it (um) well we have a few (uh) dietary restrictions some people are vegan and a couple
ASR PREDICTION excerpt: ring ring hi there this is Leo from feast and co your personal lunch superhero am I speaking with Ashley hi Leo yeah this is Ashley I'm looking into corporate catering for a team lunch oh I love a good team lunch nothing brings co-workers together like gourmet bites and good vibes so tell me what kind of flavor adventure are we talking about here Wow you sound very enthusiastic about this I love it well we have a few die

,start,end,word,probability
0,11.44,12.12,ring,0.0675
1,12.12,12.52,ring,0.8451
2,12.52,12.92,hi,0.2952
3,12.92,13.32,there,0.9811
4,13.32,13.82,this,0.6402
5,13.82,14.04,is,0.8519
6,14.04,14.36,Leo,0.3442
7,14.36,14.98,from,0.9829
8,14.98,15.44,feast,0.8388
9,15.44,15.66,and,0.8481


## 7. Save run provenance and validate outputs

In this version, transcription **does not assign Customer/Agent roles**. Reference roles are available but not copied into the model predictions. This distinction is important when later presenting WER, diarization performance and limitations to reviewers.

In [20]:
import importlib.metadata as package_metadata
run_config = {
    "stage": "02_transcription",
    "dataset": REPO_ID,
    "dataset_config": CONFIG,
    "dataset_revision": DATASET_REVISION,
    "model": MODEL_REPO,
    "model_revision": model_revision,
    "device": DEVICE,
    "compute_type": COMPUTE_TYPE,
    "asr_settings": ASR_SETTINGS,
    "seed": SEED,
    "pilot_only": PILOT_ONLY,
    "processed_calls": len(metrics_df),
    "sample_manifest": str(manifest_path),
    "versions": {
        "python": platform.python_version(),
        "faster-whisper": package_metadata.version("faster-whisper"),
        "jiwer": package_metadata.version("jiwer"),
        "huggingface_hub": package_metadata.version("huggingface_hub"),
    },
    "normalization": "NFKC + lowercase + normalize apostrophe + remove non-[a-z0-9'\\s] characters + collapse spaces; not dataset official scorer",
    "reference_usage": "Reference transcripts loaded ONLY AFTER ASR inference for WER/CER evaluation",
    "speaker_diarization": "NOT PERFORMED IN THIS NOTEBOOK; no predicted customer/agent roles",
    "license_note": "AppTek CC BY-SA 4.0; role-played calls, evaluation/analysis use, do not redistribute WAVs in GitHub",
}
(ASR_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False), encoding="utf-8")
assert metrics_df.call_id.nunique() == len(to_process)
assert metrics_df.wer.notna().all() and metrics_df.cer.notna().all()
assert all((ASR_DIR / f"{call_id}.json").is_file() for call_id in metrics_df.call_id)
print("✓ Transcription and evaluation completed for", len(metrics_df), "calls")
print("✓ Saved results:", ASR_DIR.resolve())
print("✓ Corpus WER:", round(jiwer.wer(normalized_refs, normalized_hyps), 4))
print("NEXT: Share the WER/CER table and 2 excerpts for review. Only then set PILOT_ONLY=False to process all 12 calls.")

✓ Transcription and evaluation completed for 12 calls
✓ Saved results: /content/data/processed/asr/small_en_cpu
✓ Corpus WER: 0.2075
NEXT: Share the WER/CER table and 2 excerpts for review. Only then set PILOT_ONLY=False to process all 12 calls.


## Experiment: Impact of VAD on ASR Performance

### Objective
Investigate whether disabling Voice Activity Detection (VAD)
reduces transcription errors, particularly deletions, in CALL_002.

### Experimental Setup
- Model: Whisper Small.en
- Device: CPU (INT8)
- Test call: CALL_002
- Configuration A: VAD Enabled
- Configuration B: VAD Disabled
- All other ASR settings remained unchanged.

In [10]:
# Experiment: Transcription without VAD

test_row = sample[sample["call_id"] == "CALL_002"].iloc[0]

audio, sr = sf.read(
    audio_paths["CALL_002"],
    dtype="float32"
)

# Disable VAD for comparison
test_settings = ASR_SETTINGS.copy()
test_settings["vad_filter"] = False
test_settings.pop("vad_parameters", None)

start_time = time.perf_counter()

segments, info = model.transcribe(
    audio,
    **test_settings
)

predicted_text = " ".join(
    segment.text.strip()
    for segment in segments
)

elapsed = time.perf_counter() - start_time

# Get reference transcript
reference_row = lookup[
    (test_row["metadata_path"], test_row["file_name"])
]

reference_text = " ".join(
    s["text"]
    for s in sorted(
        reference_row["segments"],
        key=lambda x: (x["start"], x["end"])
    )
)

# Normalize
ref = normalize_for_metrics(reference_text)
hyp = normalize_for_metrics(predicted_text)

# Evaluate
result = jiwer.process_words(ref, hyp)

print("CALL_002 — WITHOUT VAD")
print("-----------------------")
print(f"WER: {result.wer:.3f}")
print(f"CER: {jiwer.cer(ref, hyp):.3f}")
print(f"Substitutions: {result.substitutions}")
print(f"Deletions: {result.deletions}")
print(f"Insertions: {result.insertions}")
print(f"Inference time: {elapsed:.2f} seconds")

CALL_002 — WITHOUT VAD
-----------------------
WER: 0.166
CER: 0.109
Substitutions: 50
Deletions: 204
Insertions: 14
Inference time: 192.53 seconds


### Experimental Results

| Metric | VAD Enabled | VAD Disabled |
|--------|-------------|--------------|
| WER | 16.6% | 16.6% |
| CER | 11.3% | 10.9% |
| Substitutions | 44 | 50 |
| Deletions | 211 | 204 |
| Insertions | 14 | 14 |
| Inference Time | 177.33s | 192.53s |

### Conclusion
Disabling VAD did not meaningfully improve WER on CALL_002.
Although deletions decreased slightly, substitutions increased.

Therefore, VAD was retained for the main ASR pipeline because
it achieved comparable accuracy with lower inference time.

This conclusion is based on a single-call experiment and
may not generalize to all accents or domains.

## 8. ASR Error Analysis

### Objective
Investigate the main causes of transcription errors,
particularly the high deletion rates observed in some calls.

### Selected Calls
- CALL_001: Best-performing call (WER: 8.5%)
- CALL_005: High transcription errors (WER: 30.9%)
- CALL_006: Worst-performing call (WER: 36.2%)

### Investigation
1. Analyze overlapping speech between Customer and Agent.
2. Identify time windows with potentially missing words.
3. Compare reference transcripts with ASR predictions.
4. Listen to problematic audio segments.
5. Document findings and limitations.

In [21]:
# ============================================================
# ASR Error Analysis — Speaker Overlap Investigation
# ============================================================

import json
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display

# Select calls for investigation
CALL_IDS = ["CALL_001", "CALL_005", "CALL_006"]


def calculate_overlap(segments):
    """
    Calculate the total duration of overlapping speech
    between Customer and Agent using reference annotations.
    """

    customer = [
        (float(s["start"]), float(s["end"]))
        for s in segments
        if str(s.get("role", "")).lower() == "customer"
    ]

    agent = [
        (float(s["start"]), float(s["end"]))
        for s in segments
        if str(s.get("role", "")).lower() == "agent"
    ]

    # Find overlapping intervals
    overlaps = []

    for c_start, c_end in customer:
        for a_start, a_end in agent:

            start = max(c_start, a_start)
            end = min(c_end, a_end)

            if start < end:
                overlaps.append((start, end))

    # Merge intersecting intervals to avoid double counting
    overlaps.sort()
    merged = []

    for start, end in overlaps:
        if merged and start <= merged[-1][1]:
            merged[-1] = (
                merged[-1][0],
                max(merged[-1][1], end)
            )
        else:
            merged.append((start, end))

    return sum(end - start for start, end in merged)


overlap_results = []

for call_id in CALL_IDS:

    row = sample[sample["call_id"] == call_id].iloc[0]

    reference = lookup[
        (row["metadata_path"], row["file_name"])
    ]

    segments = reference["segments"]

    overlap_seconds = calculate_overlap(segments)

    duration = float(row["duration_sec"])

    overlap_results.append({
        "call_id": call_id,
        "accent": row["accent"],
        "duration_sec": round(duration, 2),
        "overlap_sec": round(overlap_seconds, 2),
        "overlap_percent": round(
            overlap_seconds / duration * 100, 2
        ),
        "wer": float(
            metrics_df.loc[
                metrics_df["call_id"] == call_id, "wer"
            ].iloc[0]
        )
    })


overlap_df = pd.DataFrame(overlap_results)

print("Speaker Overlap Analysis")
display(overlap_df)

overlap_df.to_csv(
    ASR_DIR / "speaker_overlap_analysis.csv",
    index=False
)

Speaker Overlap Analysis


,call_id,accent,duration_sec,overlap_sec,overlap_percent,wer
0,CALL_001,en-US_General,386.64,0.59,0.15,0.0849
1,CALL_005,en-GB,611.11,60.71,9.93,0.3091
2,CALL_006,en-GB,603.48,35.60,5.90,0.3623


In [22]:
# ============================================================
# ASR Error Analysis — Missing Speech Investigation
# ============================================================

WINDOW_SECONDS = 30

window_results = []

for call_id in CALL_IDS:

    row = sample[sample["call_id"] == call_id].iloc[0]

    reference = lookup[
        (row["metadata_path"], row["file_name"])
    ]

    prediction = json.loads(
        (ASR_DIR / f"{call_id}.json").read_text(
            encoding="utf-8"
        )
    )

    ref_segments = reference["segments"]
    pred_segments = prediction["predicted_segments"]

    duration = float(row["duration_sec"])

    for start in range(0, int(duration), WINDOW_SECONDS):

        end = min(start + WINDOW_SECONDS, duration)

        # Assign each segment to a window by its midpoint.
        # This is a diagnostic approximation, not official WER.
        reference_window = [
            s for s in ref_segments
            if start <= (
                float(s["start"]) + float(s["end"])
            ) / 2 < end
        ]

        predicted_window = [
            s for s in pred_segments
            if start <= (
                float(s["start"]) + float(s["end"])
            ) / 2 < end
        ]

        ref_text = " ".join(
            str(s.get("text", ""))
            for s in sorted(
                reference_window,
                key=lambda x: float(x["start"])
            )
        )

        pred_text = " ".join(
            str(s.get("text", ""))
            for s in sorted(
                predicted_window,
                key=lambda x: float(x["start"])
            )
        )

        ref_words = len(
            normalize_for_metrics(ref_text).split()
        )

        pred_words = len(
            normalize_for_metrics(pred_text).split()
        )

        window_results.append({
            "call_id": call_id,
            "start": start,
            "end": end,
            "reference_words": ref_words,
            "predicted_words": pred_words,
            "word_difference": ref_words - pred_words,
            "reference_text": ref_text,
            "predicted_text": pred_text
        })


windows_df = pd.DataFrame(window_results)

# Show windows with the largest shortage of predicted words
worst_windows = (
    windows_df[windows_df["reference_words"] >= 10]
    .sort_values(
        "word_difference",
        ascending=False
    )
    .groupby("call_id", group_keys=False)
    .head(3)
)

print("Potential Missing Speech — Top 3 Windows Per Call")

display(
    worst_windows[
        [
            "call_id",
            "start",
            "end",
            "reference_words",
            "predicted_words",
            "word_difference"
        ]
    ]
)

windows_df.to_csv(
    ASR_DIR / "asr_error_windows.csv",
    index=False
)

Potential Missing Speech — Top 3 Windows Per Call


,call_id,start,end,reference_words,predicted_words,word_difference
31,CALL_005,540,570.0,119,76,43
50,CALL_006,480,510.0,74,31,43
51,CALL_006,510,540.0,75,36,39
46,CALL_006,360,390.0,111,74,37
23,CALL_005,300,330.0,101,69,32
4,CALL_001,120,150.0,83,58,25
18,CALL_005,150,180.0,114,90,24
3,CALL_001,90,120.0,69,47,22
10,CALL_001,300,330.0,82,64,18


In [23]:
# ============================================================
# Manual Audio Inspection
# ============================================================

for call_id in CALL_IDS:

    selected = worst_windows[
        worst_windows["call_id"] == call_id
    ].head(1)

    if selected.empty:
        continue

    row = selected.iloc[0]

    start = float(row["start"])
    end = float(row["end"])

    print("=" * 80)
    print(f"CALL: {call_id}")
    print(f"TIME: {start:.0f}s - {end:.0f}s")

    print("\nREFERENCE TRANSCRIPT:")
    print(row["reference_text"])

    print("\nWHISPER TRANSCRIPT:")
    print(row["predicted_text"])

    print("\nORIGINAL AUDIO:")

    audio_info = sf.info(audio_paths[call_id])
    sr = audio_info.samplerate

    audio, _ = sf.read(
        audio_paths[call_id],
        start=int(start * sr),
        stop=int(end * sr),
        dtype="float32"
    )

    display(Audio(audio, rate=sr))

CALL: CALL_001
TIME: 120s - 150s

REFERENCE TRANSCRIPT:
What about delivery (uh) do you think you can handle (uh) set up too? Yeah absolutely (um) we'll deliver your feast right to your office set it up beautifully, make sure it looks like a million bucks, napkins folded, plates staggered. All of the works you only need to focus on bringing your appetite. Wow (um) that's amazing, okay cool. (uh) can you send me maybe a sample menu (um) just, you know but with the ones that you as well told me.

WHISPER TRANSCRIPT:
What about delivery? Do you think you can handle a set up too? Yeah, absolutely. We'll deliver your feast right to your office, set it up beautifully, make sure it looks like a million bucks, napkins, folded plates, staggered, all the works. You only need to focus on bringing your appetite. Wow. That's amazing. Okay, cool. Can you

ORIGINAL AUDIO:


CALL: CALL_005
TIME: 540s - 570s

REFERENCE TRANSCRIPT:
And (um,) we do have, (um,) special offers of both of you in store and online. (uh) would be interested if we were to send you our online catalog? Yeah, thank you. Yea, ohh I'd very much love that. Yes, so online catalog it's been added. I've sent this to you~ your email now, and just to confirm your email again, please Yeah, yep (um,) my nam~ email is (uh) honey at gmail dot com. Honey at Gmail dot com. I've sent you that now, and I've added your daughter into the loyalty scheme card. (um) currently, I've reduced, (uh,) you wanted to use your fifteen pound voucher that you had on there while the total comes up to.

WHISPER TRANSCRIPT:
Yeah, thank you. Would you be interested if we were to send you our online catalogue? Oh, I'd very much love that. Yeah, so online catalogue has been added. I've sent this to your email now. And just to confirm your email again, please. yep and my name my email is honey at gmail.com honey at gmail.c

CALL: CALL_006
TIME: 480s - 510s

REFERENCE TRANSCRIPT:
Zero seven eight two six five four three. Yeah. Yeah. Thank you. (Um) would you like a text message confirmation? Could I have both (um) text message and email please? Sure. We do. (Um) we would recommend that you (um) put (uh) you and your guests will put your (um) (uh) phone on airplane mode on entrance. Will that be a problem on that day? Ohh. Yeah. Yes. That's not a problem. Thank you.

WHISPER TRANSCRIPT:
Just confirmation. Eight six. Yeah. Five four three. Yeah. Thank you. Sure. We do. We would recommend that you put you and your guests will put your phone on airplane mode

ORIGINAL AUDIO:


## 9. Filler-Adjusted WER Evaluation

### Objective
Evaluate the impact of filler words (um, uh) on ASR accuracy.

The original WER is preserved as the primary evaluation metric.

An additional WER is calculated after removing common
filler words from both reference and predicted transcripts.

This experiment helps distinguish transcription errors
from the omission of non-essential hesitation words.

In [24]:
# ============================================================
# Filler-Adjusted WER Evaluation
# ============================================================

import re
import json
import pandas as pd
import jiwer

def remove_fillers(text):
    """
    Remove common hesitation words from normalized text.
    """
    text = re.sub(r"\b(um|uh)\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()


results = []

original_refs = []
original_hyps = []

adjusted_refs = []
adjusted_hyps = []

for row in to_process.itertuples(index=False):

    call_id = row.call_id

    # Reference transcript
    reference_row = lookup[
        (row.metadata_path, row.file_name)
    ]

    reference_text = " ".join(
        s["text"]
        for s in sorted(
            reference_row["segments"],
            key=lambda x: (x["start"], x["end"])
        )
    )

    # Whisper prediction
    prediction = json.loads(
        (ASR_DIR / f"{call_id}.json").read_text(
            encoding="utf-8"
        )
    )

    predicted_text = prediction["predicted_transcript"]

    # Standard normalization
    ref = normalize_for_metrics(reference_text)
    hyp = normalize_for_metrics(predicted_text)

    # Remove filler words
    ref_adjusted = remove_fillers(ref)
    hyp_adjusted = remove_fillers(hyp)

    # Calculate WER
    original_wer = jiwer.wer(ref, hyp)

    adjusted_wer = jiwer.wer(
        ref_adjusted,
        hyp_adjusted
    )

    original_refs.append(ref)
    original_hyps.append(hyp)

    adjusted_refs.append(ref_adjusted)
    adjusted_hyps.append(hyp_adjusted)

    results.append({
        "call_id": call_id,
        "original_wer": round(original_wer, 4),
        "adjusted_wer": round(adjusted_wer, 4),
        "improvement_pp": round(
            (original_wer - adjusted_wer) * 100, 2
        )
    })


comparison_df = pd.DataFrame(results)

print("WER Comparison: Original vs Filler-Adjusted")

display(comparison_df)

print("\nCORPUS RESULTS")
print("-" * 40)

print(
    "Original Corpus WER:",
    round(jiwer.wer(original_refs, original_hyps), 4)
)

print(
    "Filler-Adjusted Corpus WER:",
    round(jiwer.wer(adjusted_refs, adjusted_hyps), 4)
)

comparison_df.to_csv(
    ASR_DIR / "filler_adjusted_wer.csv",
    index=False
)

WER Comparison: Original vs Filler-Adjusted


,call_id,original_wer,adjusted_wer,improvement_pp
0,CALL_001,0.0849,0.0549,3.00
1,CALL_002,0.1665,0.1265,4.00
2,CALL_003,0.2569,0.2405,1.64
3,CALL_004,0.1532,0.1430,1.03
4,CALL_005,0.3091,0.2800,2.91
5,CALL_006,0.3623,0.3069,5.53
6,CALL_007,0.2799,0.2788,0.11
7,CALL_008,0.2387,0.2208,1.79
8,CALL_009,0.2551,0.2147,4.04
9,CALL_010,0.1544,0.1400,1.44



CORPUS RESULTS
----------------------------------------
Original Corpus WER: 0.2075
Filler-Adjusted Corpus WER: 0.1802


## 10. Experiment: Chunked vs. Full-Audio Transcription

### Objective
Investigate whether processing long audio recordings in
shorter chunks improves ASR accuracy.

### Experimental Setup
- Model: Whisper Small.en
- Device: CPU (INT8)
- Test call: CALL_006
- Baseline: Full-audio transcription
- Experiment: 60-second audio chunks
- ASR settings: Unchanged

### Evaluation
Compare WER, filler-adjusted WER, and inference time.

Note: Fixed-length chunking may split words or sentences
at chunk boundaries. Results are exploratory and will
not replace the original baseline without further validation.

In [25]:
# ============================================================
# Experiment: Chunked vs Full-Audio Transcription
# ============================================================

import time
import json
import soundfile as sf
import jiwer

CALL_ID = "CALL_006"
CHUNK_SECONDS = 60

# Load original audio
audio, sr = sf.read(
    audio_paths[CALL_ID],
    dtype="float32"
)

assert sr == 16000
assert audio.ndim == 1

chunk_samples = CHUNK_SECONDS * sr

chunk_transcripts = []

start_time = time.perf_counter()

# Transcribe audio in 60-second chunks
for start in range(0, len(audio), chunk_samples):

    end = min(start + chunk_samples, len(audio))

    chunk = audio[start:end]

    segments, info = model.transcribe(
        chunk,
        **ASR_SETTINGS
    )

    text = " ".join(
        segment.text.strip()
        for segment in segments
    )

    chunk_transcripts.append(text)

elapsed = time.perf_counter() - start_time

# Combine predicted transcripts
chunked_transcript = " ".join(chunk_transcripts)

# Load reference transcript
row = sample[sample["call_id"] == CALL_ID].iloc[0]

reference_row = lookup[
    (row["metadata_path"], row["file_name"])
]

reference_text = " ".join(
    str(s["text"])
    for s in sorted(
        reference_row["segments"],
        key=lambda x: (x["start"], x["end"])
    )
)

# Load original Whisper prediction
original_prediction = json.loads(
    (ASR_DIR / f"{CALL_ID}.json").read_text(
        encoding="utf-8"
    )
)

original_text = original_prediction["predicted_transcript"]

# Normalize
ref = normalize_for_metrics(reference_text)
original = normalize_for_metrics(original_text)
chunked = normalize_for_metrics(chunked_transcript)

# Standard WER
original_wer = jiwer.wer(ref, original)
chunked_wer = jiwer.wer(ref, chunked)

# Filler-adjusted WER
ref_adjusted = remove_fillers(ref)
original_adjusted = remove_fillers(original)
chunked_adjusted = remove_fillers(chunked)

original_filler_wer = jiwer.wer(
    ref_adjusted,
    original_adjusted
)

chunked_filler_wer = jiwer.wer(
    ref_adjusted,
    chunked_adjusted
)

print("=" * 55)
print("CALL_006 — ASR CHUNKING EXPERIMENT")
print("=" * 55)

print(f"Original WER:          {original_wer:.2%}")
print(f"Chunked WER:           {chunked_wer:.2%}")

print()

print(f"Original Adjusted WER: {original_filler_wer:.2%}")
print(f"Chunked Adjusted WER:  {chunked_filler_wer:.2%}")

print()

print(f"Chunked Inference Time: {elapsed:.2f} seconds")

print("=" * 55)

# Save experiment results without overwriting baseline
experiment_dir = ASR_DIR / "experiments"
experiment_dir.mkdir(parents=True, exist_ok=True)

experiment_result = {
    "call_id": CALL_ID,
    "chunk_seconds": CHUNK_SECONDS,
    "original_wer": original_wer,
    "chunked_wer": chunked_wer,
    "original_adjusted_wer": original_filler_wer,
    "chunked_adjusted_wer": chunked_filler_wer,
    "chunked_inference_seconds": elapsed,
    "chunked_transcript": chunked_transcript
}

output_file = experiment_dir / "CALL_006_chunking.json"

output_file.write_text(
    json.dumps(experiment_result, indent=2),
    encoding="utf-8"
)

print("Results saved:", output_file)

CALL_006 — ASR CHUNKING EXPERIMENT
Original WER:          36.23%
Chunked WER:           35.83%

Original Adjusted WER: 30.69%
Chunked Adjusted WER:  31.20%

Chunked Inference Time: 188.50 seconds
Results saved: data/processed/asr/small_en_cpu/experiments/CALL_006_chunking.json


### Chunking Experiment Results

| Metric | Full Audio | 60s Chunks |
|--------|------------|------------|
| WER | 36.23% | 35.83% |
| Filler-Adjusted WER | 30.69% | 31.20% |
| Inference Time | 143.41s | 188.50s |

### Findings

Chunking produced only a marginal improvement in standard
WER (0.40 percentage points), while filler-adjusted WER
increased by 0.51 percentage points.

Processing time also increased by approximately 31.4%.

### Conclusion

The experiment did not demonstrate a meaningful benefit
from fixed-length chunking on CALL_006.

Therefore, full-audio transcription with VAD enabled was
retained as the baseline ASR approach.

Further investigation may be required to understand the
effects of audio quality and overlapping speech.